## mpi4py

### Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [One Program, Many Processes](#2-one-program-many-processes)
3. [Exercise 1: Distributed Sum of Squares](#3-exercise-1-distributed-sum-of-squares)
4. [Domain Decomposition and Halo Exchange](#4-domain-decomposition-and-halo-exchange)
5. [The Serial Heat Equation Baseline](#5-the-serial-heat-equation-baseline)
6. [Exercise 2: Distributed Heat Equation](#6-exercise-2-distributed-heat-equation)
7. [Verification and Visualization](#7-verification-and-visualization)
8. [Key Takeaways](#8-key-takeaways)

---

### 1. Environment Setup

[MPI](https://www.mpi-forum.org/) (Message Passing Interface) lets independent processes cooperate by sending data to one another. [mpi4py](https://mpi4py.readthedocs.io/en/stable/) exposes MPI in Python while supporting efficient communication directly from NumPy arrays.

In this notebook, we'll learn to:

1. Launch the same Python program as several MPI processes.
2. Combine distributed results with a collective operation.
3. Split a two-dimensional grid into row-wise subdomains.
4. Exchange halo rows between neighboring processes.
5. Verify a distributed heat-equation stencil against a serial reference.

First, let's make sure the MPI launcher and Python environment are ready. The setup selects MPICH's local `fork` launcher in the CSCS environment and Open MPI's oversubscription mode in Colab or a local tutorial environment. MPI does not use the GPU in this lesson; the standard GPU notebook environment is retained so this notebook composes with the rest of the tutorial.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Install Open MPI and mpi4py if running in Google Colab.
colab_marker = Path("/tmp/accelerated-computing-hub-mpi4py-installed")
if os.getenv("COLAB_RELEASE_TAG") and not colab_marker.exists():
    print("Installing Open MPI and mpi4py.")
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "openmpi-bin", "libopenmpi-dev"],
        check=True,
        stdout=subprocess.DEVNULL,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "mpi4py"],
        check=True,
        stdout=subprocess.DEVNULL,
    )
    colab_marker.touch()
    print("Open MPI and mpi4py installed.")

# Open MPI protects against accidental root launches. Colab and the tutorial
# container are isolated environments where explicitly allowing this is safe.
os.environ.setdefault("OMPI_ALLOW_RUN_AS_ROOT", "1")
os.environ.setdefault("OMPI_ALLOW_RUN_AS_ROOT_CONFIRM", "1")

# Match the launcher to the MPI library against which mpi4py was built.
vendor_result = subprocess.run(
    [
        sys.executable,
        "-c",
        "from mpi4py import MPI; print(MPI.get_vendor()[0])",
    ],
    check=True,
    capture_output=True,
    text=True,
)
MPI_VENDOR = vendor_result.stdout.strip()

if MPI_VENDOR == "MPICH":
    mpi_launcher = (
        shutil.which("mpirun.mpich")
        or shutil.which("mpiexec.mpich")
        or shutil.which("mpiexec")
    )
    if mpi_launcher is None:
        raise RuntimeError("No MPICH launcher was found")
    # Do not delegate this nested launch back to Slurm on CSCS.
    MPI_LAUNCHER = [mpi_launcher, "-launcher", "fork"]
elif MPI_VENDOR == "Open MPI":
    mpi_launcher = shutil.which("mpirun.openmpi") or shutil.which("mpirun")
    if mpi_launcher is None:
        raise RuntimeError("No Open MPI launcher was found")
    MPI_LAUNCHER = [mpi_launcher, "--oversubscribe"]
else:
    mpi_launcher = shutil.which("mpiexec")
    if mpi_launcher is None:
        raise RuntimeError(f"No launcher was found for {MPI_VENDOR}")
    MPI_LAUNCHER = [mpi_launcher]


def run_program(command):
    '''Run a child program, display its output, and fail on errors.'''
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=180,
    )
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr)
    result.check_returncode()


def run_mpi(rank_count, script):
    '''Run a Python script under the selected MPI implementation.'''
    command = [
        *MPI_LAUNCHER,
        "-n",
        str(rank_count),
        sys.executable,
        "-u",
        script,
    ]
    run_program(command)


def run_python(script):
    '''Run a serial Python reference with the notebook's interpreter.'''
    run_program([sys.executable, "-u", script])

### 2. One Program, Many Processes

MPI follows the **single program, multiple data** model: every process runs the same script, but each process has a different integer **rank**. The processes belong to a **communicator**; `MPI.COMM_WORLD` contains every process launched for the program.

The notebook kernel itself is a single process, so MPI examples need to be written to Python files and launched with `mpirun`. Passing `4` to the `run_mpi` helper below adds `-n 4` to the launcher and creates four independent Python processes. Each process discovers:

- its own rank with `comm.Get_rank()`;
- the communicator size with `comm.Get_size()`; and
- its host name with `MPI.Get_processor_name()`.

Calling `gather` is a **collective operation**: every rank participates, and rank 0 receives one value from each rank. Gathering the values before printing makes the rank order deterministic, even though host names depend on the system running the notebook.

In [ ]:
%%writefile hello_mpi.py

from mpi4py import MPI


comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

message = f"rank {rank} of {size} on {MPI.Get_processor_name()}"
messages = comm.gather(message, root=0)

if rank == 0:
    print(f"Launched {size} MPI processes:")
    for message in messages:
        print(f"  {message}")

In [ ]:
run_mpi(4, "hello_mpi.py")

### 3. Exercise 1: Distributed Sum of Squares

Our first collective gathered Python strings. Now let's distribute numerical work. Rank 0 creates the integers 1 through 23 and `scatter` sends one NumPy chunk to each rank. The chunks may have different lengths, which the lowercase object-based `scatter` method handles naturally.

**Simple exercise:** Each rank already computes its local sum of squares. Your task is to combine those partial results on rank 0.

**TODO:** Replace the placeholder with one call to `comm.reduce`. Use `op=MPI.SUM` and `root=0`.

The unchanged validation checks both the one-rank and four-rank results independently against NumPy.

In [ ]:
%%writefile distributed_sum_squares.py

import numpy as np
from mpi4py import MPI


comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

values = np.arange(1, 24, dtype=np.int64) if rank == 0 else None
chunks = np.array_split(values, size) if rank == 0 else None
local_values = comm.scatter(chunks, root=0)

local_sum_squares = np.dot(local_values, local_values)

# TODO: Sum the partial results on rank 0 with comm.reduce.
raise NotImplementedError("TODO: reduce the local sums")

if rank == 0:
    expected = np.dot(values, values)
    assert global_sum_squares == expected
    print(f"sum(i**2 for i in 1..23) = {global_sum_squares}")

In [ ]:
run_mpi(1, "distributed_sum_squares.py")
run_mpi(4, "distributed_sum_squares.py")

#### Think About It

What does `global_sum_squares` contain on ranks other than rank 0? When would `allreduce` be more appropriate than `reduce`?

### 4. Domain Decomposition and Halo Exchange

Collectives move data across an entire communicator. Stencil computations have a more local communication pattern: each grid point depends only on nearby points. We can split a $66 \times 66$ grid by rows and give each of four ranks 16 of the 64 interior rows:

```text
global row:   0 |  1 ... 16 | 17 ... 32 | 33 ... 48 | 49 ... 64 | 65
owner: boundary |   rank 0  |   rank 1  |   rank 2  |   rank 3  | boundary
```

Every rank stores two extra **halo rows** around its owned rows:

```text
              top halo
          +----------------+
          |   owned rows   |
          +----------------+
             bottom halo
```

Before each stencil update, adjacent ranks exchange their edge rows. `MPI.PROC_NULL` represents a missing neighbor at a physical boundary: communication with it completes immediately and leaves the receive buffer unchanged. Using `Sendrecv` pairs a send and receive in one operation, avoiding the deadlock risks of separate blocking sends.

For NumPy arrays, mpi4py's uppercase methods such as `Sendrecv` and `Gather` communicate typed buffers directly. The lowercase methods used above can serialize general Python objects; uppercase methods are the natural choice for fixed-shape numerical arrays.

### 5. The Serial Heat Equation Baseline

The two-dimensional heat equation is

$$
\frac{\partial u}{\partial t}
= \kappa \left(\frac{\partial^2 u}{\partial x^2}
+ \frac{\partial^2 u}{\partial y^2}\right).
$$

For equal grid spacing $h = \Delta x = \Delta y$, define the dimensionless diffusion number

$$r = \frac{\kappa\,\Delta t}{h^2}.$$

A five-point finite-difference stencil then advances one time step:

$$
u^{n+1}_{i,j} = u^n_{i,j} + r\left(
u^n_{i-1,j} + u^n_{i+1,j} + u^n_{i,j-1} + u^n_{i,j+1}
- 4u^n_{i,j}\right).
$$

We use $r=0.2$, below the two-dimensional explicit stability limit $r \leq 1/4$, and hold the outer boundary at zero. The complete serial implementation below gives us a trusted reference before communication is introduced.

In [ ]:
%%writefile heat_reference.py

import numpy as np


NY = 66
NX = 66
STEPS = 120
DIFFUSION_NUMBER = 0.2


def initial_temperature_rows(global_rows, ny=NY, nx=NX):
    """Create the initial hot disk for selected global rows."""
    y = np.asarray(global_rows)[:, np.newaxis]
    x = np.arange(nx)[np.newaxis, :]
    center_y = (ny - 1) / 2
    center_x = (nx - 1) / 2
    hot = (y - center_y) ** 2 + (x - center_x) ** 2 <= 6**2
    temperature = hot.astype(np.float64)
    temperature[:, 0] = 0.0
    temperature[:, -1] = 0.0
    return temperature


def initial_temperature(ny=NY, nx=NX):
    """Create the full initial field with zero-valued boundaries."""
    temperature = np.zeros((ny, nx), dtype=np.float64)
    temperature[1:-1] = initial_temperature_rows(range(1, ny - 1), ny, nx)
    return temperature


def advance(temperature, diffusion_number=DIFFUSION_NUMBER):
    """Apply one serial five-point stencil update."""
    next_temperature = np.zeros_like(temperature)
    center = temperature[1:-1, 1:-1]
    next_temperature[1:-1, 1:-1] = center + diffusion_number * (
        temperature[:-2, 1:-1]
        + temperature[2:, 1:-1]
        + temperature[1:-1, :-2]
        + temperature[1:-1, 2:]
        - 4.0 * center
    )
    return next_temperature


def solve_serial(steps=STEPS):
    """Evolve the complete grid on one process."""
    temperature = initial_temperature()
    for _ in range(steps):
        temperature = advance(temperature)
    return temperature


if __name__ == "__main__":
    temperature = solve_serial()
    assert np.isfinite(temperature).all()
    assert np.all(temperature >= 0.0)
    assert np.all(temperature <= 1.0)
    assert np.all(temperature[[0, -1], :] == 0.0)
    assert np.all(temperature[:, [0, -1]] == 0.0)
    print(
        f"Serial reference passed: shape={temperature.shape}, "
        f"maximum={temperature.max():.6f}"
    )

In [ ]:
run_python("heat_reference.py")

### 6. Exercise 2: Distributed Heat Equation

The distributed solver retains the serial stencil but stores only one row slab per rank.

**Advanced exercise:** Complete its two missing pieces:

1. **Exchange halo rows.** Add two `comm.Sendrecv` calls to `exchange_halos`:
   - send the first owned row (`temperature[1]`) to `above` while receiving the bottom halo (`temperature[-1]`) from `below`, using tag 0;
   - send the last owned row (`temperature[-2]`) to `below` while receiving the top halo (`temperature[0]`) from `above`, using tag 1.
2. **Advance the local stencil.** Fill `next_temperature[1:-1, 1:-1]` with the same five-point update as the serial baseline. The halo rows make the slice expression identical even at rank boundaries.

**TODO:** Fill both placeholders and remove their `NotImplementedError` lines. The communication calls use this keyword form:

```python
comm.Sendrecv(
    sendbuf=..., dest=..., sendtag=...,
    recvbuf=..., source=..., recvtag=...,
)
```

The decomposition, initialization, gather, and serial-reference check are already complete.

In [ ]:
%%writefile heat_mpi.py

import numpy as np
from mpi4py import MPI

from heat_reference import (
    DIFFUSION_NUMBER,
    NX,
    NY,
    STEPS,
    initial_temperature,
    initial_temperature_rows,
    solve_serial,
)


def decompose_rows(comm):
    """Return the local row count and first owned global row."""
    interior_rows = NY - 2
    if interior_rows % comm.Get_size() != 0:
        raise ValueError("The interior row count must be divisible by the rank count")
    local_rows = interior_rows // comm.Get_size()
    first_global_row = 1 + comm.Get_rank() * local_rows
    return local_rows, first_global_row


def exchange_halos(temperature, comm):
    """Exchange edge rows with the ranks above and below."""
    rank = comm.Get_rank()
    above = rank - 1 if rank > 0 else MPI.PROC_NULL
    below = rank + 1 if rank + 1 < comm.Get_size() else MPI.PROC_NULL

    # TODO: Exchange the first owned row for the bottom halo (tag 0),
    # then the last owned row for the top halo (tag 1).
    raise NotImplementedError("TODO: exchange halo rows")


def advance_local(temperature):
    """Apply one five-point update to the locally owned rows."""
    next_temperature = np.zeros_like(temperature)

    # TODO: Apply the stencil to next_temperature[1:-1, 1:-1].
    raise NotImplementedError("TODO: implement the local stencil")

    return next_temperature


def gather_field(local_temperature, local_rows, comm):
    """Gather equal-sized row slabs and restore the physical boundaries."""
    owned_rows = np.ascontiguousarray(local_temperature[1:-1])
    gathered = None
    if comm.Get_rank() == 0:
        gathered = np.empty((comm.Get_size(), local_rows, NX), dtype=np.float64)

    comm.Gather(owned_rows, gathered, root=0)

    if comm.Get_rank() != 0:
        return None

    temperature = np.zeros((NY, NX), dtype=np.float64)
    temperature[1:-1] = gathered.reshape(NY - 2, NX)
    return temperature


def main():
    comm = MPI.COMM_WORLD
    local_rows, first_global_row = decompose_rows(comm)
    global_rows = np.arange(first_global_row, first_global_row + local_rows)

    temperature = np.zeros((local_rows + 2, NX), dtype=np.float64)
    temperature[1:-1] = initial_temperature_rows(global_rows)

    for _ in range(STEPS):
        exchange_halos(temperature, comm)
        temperature = advance_local(temperature)

    distributed = gather_field(temperature, local_rows, comm)

    if comm.Get_rank() == 0:
        reference = solve_serial()
        np.testing.assert_allclose(distributed, reference, rtol=1e-13, atol=1e-13)
        assert np.isfinite(distributed).all()
        assert np.all(distributed[[0, -1], :] == 0.0)
        assert np.all(distributed[:, [0, -1]] == 0.0)
        np.savez(
            "heat_equation_result.npz",
            initial=initial_temperature(),
            final=distributed,
            steps=STEPS,
        )
        print(
            "Distributed heat equation matches the serial reference: "
            f"shape={distributed.shape}, maximum={distributed.max():.6f}"
        )


if __name__ == "__main__":
    main()

### 7. Verification and Visualization

Correctness comes before interpretation. The MPI program gathers the distributed row slabs on rank 0, compares every value with `solve_serial()` using `numpy.testing.assert_allclose`, checks the fixed boundaries, and only then saves the initial and final fields for visualization.

Let's run the advanced exercise with the four ranks used by our decomposition.

In [ ]:
result_path = Path("heat_equation_result.npz")
result_path.unlink(missing_ok=True)
run_mpi(4, "heat_mpi.py")
assert result_path.exists()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

with np.load("heat_equation_result.npz") as result:
    initial = result["initial"]
    final = result["final"]
    steps = int(result["steps"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for axis, field, title in zip(
    axes,
    (initial, final),
    ("Initial temperature", f"After {steps} steps"),
):
    image = axis.imshow(field, origin="lower", cmap="inferno", vmin=0.0, vmax=1.0)
    axis.set_title(title)
    axis.set_xlabel("x")
    axis.set_ylabel("y")

fig.colorbar(image, ax=axes, label="temperature")
plt.show()

### 8. Key Takeaways

- Every MPI rank runs the same program with private memory and a distinct rank.
- Collectives such as `scatter`, `reduce`, and `Gather` coordinate all ranks in a communicator.
- A row-wise decomposition turns a global stencil into local NumPy work plus two neighbor exchanges.
- Halo rows make the local stencil expression match the serial expression.
- `Sendrecv` and `MPI.PROC_NULL` express boundary-safe neighbor communication without extra synchronization.
- Comparing against a small trusted serial implementation catches communication and indexing mistakes.

The same decomposition pattern extends to larger grids, uneven partitions with `Gatherv`, and higher-dimensional process topologies.